# Capstone â€” mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Nayak-D/FlyRank---Internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** â€” each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

FlyRank content teams need to decide which pages to review first when a large portfolio contains more possible refreshes than a strategist can inspect in one cycle. The research question is: **can observable page signals produce a more useful ranked review queue than a transparent hand-written rule?**

The supported decision is prioritization for human review: inspect, refresh, expand, monitor, or leave a page alone. This is not a system for autonomous publishing, causal impact estimation, or prediction of Google's ranking algorithm.

In [1]:
study_question = "Can observable page signals improve a limited-capacity content review queue?"
decision_supported = "Prioritize pages for review"
assert study_question and decision_supported
print(study_question)
print(decision_supported)

Can observable page signals improve a limited-capacity content review queue?
Prioritize pages for review


## 2. Data

This study uses the **FlyRank ML Internship starter release**, a public-safe extract of 30,000 content-page rows. The paper uses page-level visibility, engagement, freshness, and content-shape signals. It does not expose client names, domains, titles, keywords, private queries, credentials, or raw private exports.

- **Rows scored:** 30,000
- **Target:** `is_declining_label`
- **Declining-label rate:** 0.542
- **Validation group:** client identifier, used only to create grouped holdout splits
- **Time interpretation:** the label is a current-window proxy for observed declining behavior, not a genuine future outcome
- **Excluded from features:** target-derived decision flags and pseudonymous identifiers

The public artifact credits the FlyRank AI ML Internship dataset at [flyrank.ai](https://flyrank.ai).

In [2]:
data_summary = {
    "rows_scored": 30000,
    "target": "is_declining_label",
    "declining_label_rate": 0.542,
    "validation": "client-grouped holdout",
    "public_safe": True,
}
assert data_summary["rows_scored"] == 30000
assert 0 < data_summary["declining_label_rate"] < 1
assert data_summary["public_safe"]
data_summary

{'rows_scored': 30000,
 'target': 'is_declining_label',
 'declining_label_rate': 0.542,
 'validation': 'client-grouped holdout',
 'public_safe': True}

## 3. Methodology

### Signal audit and baseline

Candidate signals were checked before modeling. Days since update had a mixed relationship with decline across buckets, while visibility combined with weak engagement created a more actionable signal than age alone. The transparent baseline ranks pages with:

`score = visibility × freshness risk × engagement risk`

### Learned model

The selected model is a class-balanced Random Forest with 300 trees and maximum depth 8. It uses observable, pre-decision features:

- `days_with_impressions`
- `log_impressions_90d`
- `avg_position`
- `content_age_days`
- `char_count` and `word_count`
- `log_clicks_90d`
- `ctr` and `scroll_rate`
- `days_with_sessions`

### Validation and leakage control

Evaluation uses a client-grouped holdout rather than a random row split, so pages from the same client portfolio do not appear in both training and evaluation. Pseudonymous IDs are used only to define groups, and target-derived decision flags are excluded from the feature set. The comparison uses the same held-out groups for the baseline and Random Forest.

In [3]:
features = [
    "days_with_impressions", "log_impressions_90d", "avg_position",
    "content_age_days", "char_count", "word_count", "log_clicks_90d",
    "ctr", "scroll_rate", "days_with_sessions",
]
validation_checks = {
    "grouped_holdout": True,
    "same_split_for_comparison": True,
    "target_derived_flags_excluded": True,
    "identifiers_used_only_for_groups": True,
}
assert len(features) == 10
assert all(validation_checks.values())
print(f"features={len(features)}")
print("validation=client-grouped holdout")

features=10
validation=client-grouped holdout


## 4. Results (vs baseline)

Both systems were evaluated on the same client-grouped holdout. The Random Forest produced a stronger top-of-queue ranking than the hand-written rule:

| System | ROC AUC | Precision@50 |
|---|---:|---:|
| Baseline rules | 0.627 | 0.240 |
| Random Forest | 0.750 | 0.740 |

The observed Precision@50 lift is **3.08x** over the baseline (`0.740 / 0.240`). In practical terms, the learned ranking placed more genuinely declining pages in the first 50 review slots on this dataset and split. The strongest recorded signals were `days_with_impressions`, `log_impressions_90d`, `avg_position`, and `content_age_days`.

This result supports a better review queue. It does not show that editing a page causes a ranking improvement, and it does not generalize beyond the evaluated data and validation design without further testing.

In [4]:
metrics = {
    "baseline_roc_auc": 0.627,
    "model_roc_auc": 0.750,
    "baseline_precision_at_50": 0.240,
    "model_precision_at_50": 0.740,
}
observed_lift = metrics["model_precision_at_50"] / metrics["baseline_precision_at_50"]
assert round(observed_lift, 2) == 3.08
assert metrics["model_precision_at_50"] > metrics["baseline_precision_at_50"]
print(f"observed Precision@50 lift={observed_lift:.2f}x")
metrics

observed Precision@50 lift=3.08x


{'baseline_roc_auc': 0.627,
 'model_roc_auc': 0.75,
 'baseline_precision_at_50': 0.24,
 'model_precision_at_50': 0.74}

## 5. Limitations

- The label is a current-window proxy, not an observed future outcome, so this is directional ranking evidence rather than a validated forecast.
- The 3.08x improvement is observed only on this dataset and client-grouped holdout; it is not a causal estimate of what refreshing a page would do.
- The system does not reflect or attempt to reverse-engineer Google's ranking algorithm.
- Thin-signal pages, especially very low-traffic pages with zero or near-zero engagement, remain difficult to rank confidently.
- The safest production use is a ranked queue for human review, not autonomous editing or publishing.
- A stronger next study would train on past-window features and evaluate a genuine next-window outcome.

In [5]:
limitations = [
    "current-window proxy label",
    "observational, not causal",
    "thin-signal pages are uncertain",
    "human review remains required",
]
assert len(limitations) == 4
print("limitations recorded:", len(limitations))

limitations recorded: 4


## 6. Ranked recommendations

1. **Generate a weekly review queue with the model ranking.** The observed Precision@50 lift gives reviewers a stronger first pass than the hand-written rule on the held-out client groups.
2. **Keep a separate thin-signal track.** Route very low-traffic or zero-engagement pages to lighter-touch review instead of forcing confidence where evidence is weak.
3. **Inspect reason codes before editing.** The model orders review opportunities; a content strategist makes the final refresh, expand, monitor, or leave-alone decision.
4. **Replace the proxy label next.** Build a future version from past-window features and a next-window outcome so the workflow can be evaluated as forecasting.

In [6]:
recommendations = [
    "Generate a weekly model-ranked review queue",
    "Route thin-signal pages separately",
    "Inspect reason codes before editing",
    "Replace the proxy label with a next-window outcome",
]
assert len(recommendations) == 4
for rank, recommendation in enumerate(recommendations, start=1):
    print(f"{rank}. {recommendation}")

1. Generate a weekly model-ranked review queue
2. Route thin-signal pages separately
3. Inspect reason codes before editing
4. Replace the proxy label with a next-window outcome


## 7. Artifacts the paper embeds

The deployed public paper is [Refresh / Content Opportunity Scoring](https://nayak-d.github.io/FlyRank---Internship/).

Supporting public-safe artifacts:

- [Source repository](https://github.com/Nayak-D/FlyRank---Internship)
- [Model report](https://github.com/Nayak-D/FlyRank---Internship/blob/main/outputs/model_report.md)
- [Refresh queue sample](https://github.com/Nayak-D/FlyRank---Internship/blob/main/outputs/refresh_queue_sample.csv)
- [Notebook directory](https://github.com/Nayak-D/FlyRank---Internship/tree/main/work/notebooks)
- [Data credit](https://flyrank.ai)

The paper embeds the top-feature-importance, action-mix, confidence-mix, trend-distribution, and reason-code charts. The page also includes the abstract, problem, data, methodology, results, limitations, recommendations, reproducibility, and acknowledgment/data-credit sections.

In [7]:
artifacts = {
    "paper_url": "https://nayak-d.github.io/FlyRank---Internship/",
    "repository": "https://github.com/Nayak-D/FlyRank---Internship",
    "model_report": "outputs/model_report.md",
    "queue_sample": "outputs/refresh_queue_sample.csv",
}
assert artifacts["paper_url"].startswith("https://")
assert artifacts["repository"].startswith("https://")
print("public artifacts recorded:", len(artifacts))

public artifacts recorded: 4


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled â€” markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime â†’ Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` â€” then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** â€” including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.


## ML-12: 5-minute showcase demo

**Question (45 seconds)**  
FlyRank content teams need to decide which pages to review first when a large portfolio contains more possible refreshes than a strategist can inspect in one cycle. The decision is whether a page should be prioritized for human review, not whether the model can predict Google's ranking algorithm.

**Method (90 seconds)**  
I compared a transparent baseline rule with a class-balanced Random Forest using 30,000 public-safe content-page rows and observable pre-decision signals such as visibility, position, freshness, engagement, and content shape. I used a client-grouped holdout so pages from the same client portfolio did not appear in both training and evaluation.

**One chart (45 seconds)**  
Show the top-50 queue precision chart: the baseline reaches 0.240 and the Random Forest reaches 0.740 on the same held-out client groups. Point out that the chart answers the operational question directly: how useful is the first review queue when reviewer capacity is limited?

**One honest result (45 seconds)**  
The model's observed Precision@50 is 3.08x the baseline on this dataset and split. This is directional ranking evidence, not a causal estimate of refresh impact and not a validated future forecast, because the label is a current-window proxy.

**One recommendation (45 seconds)**  
Use the model to generate a weekly review queue, inspect reason codes before editing, and route thin-signal pages to a separate lighter-touch track. The next research step is a true next-window outcome so the system can be evaluated as forecasting rather than prioritization.

**Close (30 seconds)**  
Invite one question about the grouped validation or the proxy label, then share the public paper and repository for the full method and limitations.

## Shareable cut: social post

I built a content-review prioritization model for the FlyRank ML internship: a transparent baseline rule versus a class-balanced Random Forest, evaluated with a client-grouped holdout on 30,000 public-safe content-page rows. The model reached Precision@50 = 0.740 versus 0.240 for the baseline, an observed 3.08x lift on this dataset and split. The useful lesson is methodological: grouped validation and an explicit proxy-label limitation make the result more credible, while the output stays a human-review queue rather than an autonomous editing decision. Full paper and evidence: https://nayak-d.github.io/FlyRank---Internship/

## Shareable cut: employer-facing summary

I built a ranked content-review engine that compares a transparent baseline with a class-balanced Random Forest to help FlyRank strategists decide which pages to inspect first. It uses 30,000 public-safe content-page rows with visibility, engagement, freshness, and content-shape signals, evaluated with a client-grouped holdout. The model reached Precision@50 = 0.740 versus 0.240 for the baseline on this dataset and split, an observed directional improvement that supports human review prioritization rather than a causal or Google-ranking claim.